In [1]:
!pip install fastapi uvicorn[standard] sqlalchemy python-multipart pandas nest_asyncio pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 517.7/517.7 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 456.8/456.8 kB 17.8 MB/s eta 0:00:00


FastAPI

In [2]:
from fastapi import FastAPI, UploadFile, File, Query
from fastapi.responses import JSONResponse
import pandas as pd
import sqlite3
from pydantic import BaseModel
from typing import List, Optional
import nest_asyncio
from pyngrok import ngrok
import uvicorn

app = FastAPI()

SQLite database connection (in memory, since Colab resets each time)

In [3]:
conn = sqlite3.connect("products.db", check_same_thread=False)
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS products (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  sku TEXT UNIQUE,
  name TEXT,
  brand TEXT,
  color TEXT,
  size TEXT,
  mrp INTEGER,
  price INTEGER,
  quantity INTEGER
)
""")
conn.commit()

Product demo data provided

In [4]:
csv_data = """sku,name,brand,color,size,mrp,price,quantity
TSHIRT-RED-001,Classic Cotton T-Shirt,StreamThreads,Red,M,799,499,20
TSHIRT-BLK-002,Classic Cotton T-Shirt,StreamThreads,Black,L,799,549,12
POLO-GRN-003,Heritage Polo,StreamThreads,Green,XL,1299,999,8
JEANS-BLU-032,Slim Fit Jeans,DenimWorks,Blue,32,1999,1599,15
JEANS-BLK-030,Slim Fit Jeans,DenimWorks,Black,30,1999,1499,18
DRESS-PNK-S,Floral Summer Dress,BloomWear,Pink,S,2499,2199,10
DRESS-YLW-M,Floral Summer Dress,BloomWear,Yellow,M,2499,1999,7
SHOE-WHT-7,Everyday Sneakers,StrideLab,White,UK7,2999,2499,25
SHOE-NVY-8,Everyday Sneakers,StrideLab,Navy,UK8,2999,2499,19
BAG-TOTE-BEI,Canvas Tote Bag,CarryCo,Beige,OneSize,899,699,35
BELT-BRN-38,Leather Belt,CarryCo,Brown,38,1199,899,40
SAREE-RED-001,Banarasi Silk Saree,Ethniq,Red,Free,6999,5999,5
KURTA-BLU-M,Cotton Kurta,Ethniq,Blue,M,1599,1299,22
JKT-OLV-L,Utility Jacket,UrbanEdge,Olive,L,3499,2999,6
TSHIRT-GRY-S,Graphic Tee,UrbanEdge,Grey,S,899,699,30
TSHIRT-WHT-XS,Graphic Tee,UrbanEdge,White,XS,899,649,14
SHIRT-CHK-M,Checked Casual Shirt,ButtonUp,Multi,M,1799,1399,16
SHIRT-PLN-L,Plain Oxford Shirt,ButtonUp,Blue,L,1899,1499,12
HOODIE-CHR-XL,Cozy Hoodie,SnugWear,Charcoal,XL,2199,1799,11
HOODIE-CRM-M,Cozy Hoodie,SnugWear,Cream,M,2199,1699,9
"""
with open("products.csv", "w") as f:
    f.write(csv_data)

In [5]:
@app.post("/upload")
async def upload_file(file: UploadFile = File(...)):
    df = pd.read_csv(file.file)
    stored, failed = 0, []

    for idx, row in df.iterrows():
        errors = []
        # Required fields
        for col in ["sku", "name", "brand", "mrp", "price"]:
            if pd.isna(row[col]) or str(row[col]).strip() == "":
                errors.append(f"{col} missing")

        # Numeric validations
        try:
            mrp = int(row["mrp"])
            price = int(row["price"])
            qty = int(row.get("quantity", 0))
        except:
            errors.append("Numeric field invalid")
            continue

        if price > mrp:
            errors.append("price > mrp")
        if qty < 0:
            errors.append("quantity negative")

        # If valid, insert to DB
        if not errors:
            try:
                cur.execute(
                    "INSERT INTO products (sku, name, brand, color, size, mrp, price, quantity) VALUES (?, ?, ?, ?, ?, ?, ?, ?)",
                    (row["sku"], row["name"], row["brand"], row.get("color",""), row.get("size",""), mrp, price, qty)
                )
                conn.commit()
                stored += 1
            except Exception as e:
                errors.append("duplicate sku")
        if errors:
            failed.append({"row": idx+2, "sku": row.get("sku", ""), "errors": errors})

    return {"stored": stored, "failed": failed}

All products (with pagination)

In [6]:
@app.get("/products")
def get_products(page: int = Query(1, ge=1), limit: int = Query(10, le=100)):
    offset = (page - 1) * limit
    cur.execute("SELECT * FROM products LIMIT ? OFFSET ?", (limit, offset))
    rows = cur.fetchall()
    return {"page": page, "limit": limit, "products": rows}

Search (by brand, color, price range)

In [7]:
@app.get("/products/search")
def search_products(brand: Optional[str] = None, color: Optional[str] = None,
                    minPrice: Optional[int] = None, maxPrice: Optional[int] = None):
    query = "SELECT * FROM products WHERE 1=1"
    params = []
    if brand:
        query += " AND brand=?"; params.append(brand)
    if color:
        query += " AND color=?"; params.append(color)
    if minPrice:
        query += " AND price>=?"; params.append(minPrice)
    if maxPrice:
        query += " AND price<=?"; params.append(maxPrice)
    cur.execute(query, tuple(params))
    rows = cur.fetchall()
    return {"results": rows}

In [9]:
!ngrok config add-authtoken 34CpQfhYIFu45eKdiC45FV3chkG_2FroMGWcCpZrPk5FpZU8W

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [14]:
from pyngrok import ngrok
import nest_asyncio
import uvicorn
from uvicorn import Config, Server

# Applying nest_asyncio so event loops don't conflict
nest_asyncio.apply()

# Connecting ngrok tunnel
public_url = ngrok.connect(8000)
print("Public URL:", public_url)

# Configuring Uvicorn server manually
config = Config(app=app, host="0.0.0.0", port=8000, log_level="info")
server = Server(config)

# Running server safely inside notebook
await server.serve()


Public URL: NgrokTunnel: "https://overdecoratively-unprotractive-iker.ngrok-free.dev" -> "http://localhost:8000"


INFO:     Started server process [436]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     2401:4900:b135:5fc1:c082:391d:4a18:6b93:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     2401:4900:b135:5fc1:c082:391d:4a18:6b93:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     2401:4900:b135:5fc1:c082:391d:4a18:6b93:0 - "POST /upload HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [436]
